# Python Course — Part 2

**Covers:** Section 4 — Advanced Functions | Section 5 — OOP | Section 6 — Intermediate Concepts

---

---
# Section 4 — Functions: Advanced

## 4.1 `*args` and `**kwargs`

### Concept

`*args` collects extra **positional** arguments into a tuple.  
`**kwargs` collects extra **keyword** arguments into a dictionary.  
Together they allow functions to accept an arbitrary number of arguments — the foundation of flexible APIs.

### Technical Deep Dive

- The names `args` and `kwargs` are conventions — the `*` and `**` are the actual syntax.
- Parameter order must be: `positional → *args → keyword-only → **kwargs`.
- You can unpack a list/tuple into positional args with `*` and a dict into keyword args with `**` at call sites.

In [ ]:
# *args — variable positional arguments
def total(*args):
    return sum(args)

print(total(1, 2, 3))           # 6
print(total(10, 20, 30, 40))    # 100

# **kwargs — variable keyword arguments
def describe(**kwargs):
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

describe(name="Alice", age=30, city="Warsaw")

In [ ]:
# Combining all parameter types
def mixed(a, b, *args, keyword_only=0, **kwargs):
    print(f"a={a}, b={b}")
    print(f"args={args}")
    print(f"keyword_only={keyword_only}")
    print(f"kwargs={kwargs}")

mixed(1, 2, 3, 4, 5, keyword_only=99, extra="yes")

In [ ]:
# Unpacking at call site
def add(x, y, z):
    return x + y + z

nums = [1, 2, 3]
config = {"x": 10, "y": 20, "z": 30}

print(add(*nums))       # unpacks list to positional args
print(add(**config))    # unpacks dict to keyword args

# Forwarding arguments — common in wrappers
def log_call(func, *args, **kwargs):
    print(f"Calling {func.__name__}")
    result = func(*args, **kwargs)
    print(f"Result: {result}")
    return result

log_call(add, 1, 2, z=3)

### Exercises

1. Write `stats(*numbers)` that returns a dict with `min`, `max`, `mean` of the given numbers.
2. Write `make_tag(tag, content, **attrs)` that returns an HTML string like `<a href="url">text</a>`.
3. Write `merge_dicts(*dicts)` that merges any number of dicts into one (later dicts override earlier ones).

### Mini Challenge

Write a function `curry(func)` that takes a two-argument function and returns a curried version: `curry(add)(1)(2)` should return `3`.

### Solutions

In [ ]:
# Exercise 1
def stats(*numbers):
    # YOUR CODE HERE
    pass
# Exercise 2
def make_tag(tag, content, **attrs):
    # YOUR CODE HERE
    pass
# Exercise 3
def merge_dicts(*dicts):
    # YOUR CODE HERE
    pass
# Mini Challenge
def curry(func):
    # YOUR CODE HERE
    pass

## 4.2 Lambda Functions

### Concept

A `lambda` is an **anonymous, single-expression function**. It is syntactic sugar for a `def` with an implicit `return`.  
Use lambdas for short, throwaway functions — especially as arguments to `sorted()`, `map()`, `filter()`.

### Technical Deep Dive

- Syntax: `lambda args: expression`
- Can capture variables from the enclosing scope (closure).
- Cannot contain statements (`if/else` blocks, `for`, assignments).
- For anything more than one line, use a named `def` — readability wins.

In [ ]:
# Basic lambda
square = lambda x: x ** 2
add = lambda x, y: x + y
greet = lambda name="World": f"Hello, {name}!"

print(square(5))
print(add(3, 4))
print(greet())
print(greet("Alice"))

In [ ]:
# Lambda with sorted() — most common use case
people = [
    {"name": "Charlie", "age": 35},
    {"name": "Alice",   "age": 30},
    {"name": "Bob",     "age": 25},
]

by_age  = sorted(people, key=lambda p: p["age"])
by_name = sorted(people, key=lambda p: p["name"])

print([p["name"] for p in by_age])
print([p["name"] for p in by_name])

# Multi-key sort
data = [("Alice", 30), ("Bob", 25), ("Alice", 25)]
print(sorted(data, key=lambda x: (x[0], x[1])))

## 4.3 Higher-Order Functions

### Concept

A **higher-order function** takes a function as an argument or returns one.  
Core built-ins: `map()`, `filter()`, `sorted()`. From `functools`: `reduce()`, `partial()`.

In [ ]:
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map() — apply function to every element, returns iterator
squares = list(map(lambda x: x**2, numbers))
print("squares:", squares)

# filter() — keep elements where function returns True
evens = list(filter(lambda x: x % 2 == 0, numbers))
print("evens:", evens)

# reduce() — fold sequence into single value
product = reduce(lambda acc, x: acc * x, numbers)
print("product:", product)    # 10! = 3628800

# Prefer comprehensions over map/filter for readability
squares_comp = [x**2 for x in numbers]          # clearer than map
evens_comp   = [x for x in numbers if x % 2 == 0]  # clearer than filter

In [ ]:
from functools import partial

# partial() — freeze some arguments of a function
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)

print(square(5))   # 25
print(cube(3))     # 27

# Practical: partial for event handlers, callbacks
def send_email(recipient, subject, body):
    print(f"To: {recipient} | Subject: {subject} | Body: {body}")

notify_admin = partial(send_email, "admin@example.com", subject="Alert")
notify_admin(body="Disk usage above 90%")

## 4.4 Closures and Factory Functions

### Concept

A **closure** is a function that **captures variables from its enclosing scope** — even after the outer function has returned.  
A **factory function** is a function that creates and returns other functions.

### Technical Deep Dive

- Closures store references to variables via `__closure__`.
- Use `nonlocal` to mutate an enclosing variable.
- Closures are the mechanism behind decorators and many design patterns.

In [ ]:
# Simple closure
def make_multiplier(factor):
    def multiply(x):
        return x * factor   # 'factor' is captured from outer scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))   # 20
print(triple(10))   # 30
print(double.__closure__[0].cell_contents)   # 2 — the captured value

In [ ]:
# Stateful closure with nonlocal
def make_counter(start=0, step=1):
    count = start
    def counter():
        nonlocal count
        current = count
        count += step
        return current
    return counter

c1 = make_counter()
c2 = make_counter(100, 10)

print(c1(), c1(), c1())      # 0 1 2
print(c2(), c2(), c2())      # 100 110 120
# Each closure has independent state
print(c1())                  # 3 — c2 didn't affect c1

## 4.5 Decorators

### Concept

A **decorator** wraps a function to extend its behavior without modifying its source code.  
The `@decorator` syntax is shorthand for `func = decorator(func)`.

### Technical Deep Dive

- Always use `functools.wraps` inside decorators to preserve the wrapped function's metadata (`__name__`, `__doc__`).
- Decorators can be stacked — applied bottom-up.
- Decorators with arguments require an extra layer of nesting.

In [ ]:
import functools
import time

# Basic decorator
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))

In [ ]:
# Decorator with arguments
def retry(max_attempts=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

@retry(max_attempts=3, exceptions=(ValueError,))
def risky_operation(x):
    import random
    if random.random() < 0.7:
        raise ValueError("Random failure")
    return f"Success with {x}"

try:
    print(risky_operation(42))
except ValueError:
    print("All attempts exhausted")

In [ ]:
# Stacked decorators
def bold(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        return f"<b>{func(*a, **kw)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        return f"<i>{func(*a, **kw)}</i>"
    return wrapper

@bold
@italic    # applied first (bottom-up)
def greet(name):
    return f"Hello, {name}"

print(greet("World"))   # <b><i>Hello, World</i></b>

## 4.6 Generators and `yield`

### Concept

A **generator** is a function that uses `yield` to produce values lazily — one at a time, on demand.  
Unlike a regular function, it pauses execution at each `yield` and resumes when `next()` is called.  
Generators are memory-efficient for large or infinite sequences.

### Technical Deep Dive

- Calling a generator function returns a **generator object** — nothing executes yet.
- `StopIteration` is raised when the function returns.
- `yield from` delegates to another iterable.
- Generators implement the **iterator protocol** (`__iter__` + `__next__`).

In [ ]:
# Basic generator
def countdown(n):
    while n > 0:
        yield n
        n -= 1

gen = countdown(5)
print(type(gen))          # <class 'generator'>
print(next(gen))          # 5
print(next(gen))          # 4
print(list(gen))          # [3, 2, 1] — consumes the rest

# Infinite generator
def integers_from(n=0):
    while True:
        yield n
        n += 1

gen = integers_from(10)
print([next(gen) for _ in range(5)])   # [10, 11, 12, 13, 14]

In [ ]:
import sys

# Memory efficiency demonstration
def read_large_file(filepath):
    """Generator that reads a file line by line — constant memory."""
    with open(filepath) as f:
        for line in f:
            yield line.rstrip()

# vs loading entire file: lines = open(filepath).readlines()

# yield from — delegate to sub-generator
def chain(*iterables):
    for it in iterables:
        yield from it

result = list(chain([1, 2], [3, 4], [5, 6]))
print(result)   # [1, 2, 3, 4, 5, 6]

# Generator pipeline
def read_numbers():
    yield from range(1, 11)

def square(nums):
    for n in nums:
        yield n ** 2

def filter_even(nums):
    for n in nums:
        if n % 2 == 0:
            yield n

pipeline = filter_even(square(read_numbers()))
print(list(pipeline))   # [4, 16, 36, 64, 100]

## 4.7 Recursion and Memoization

### Concept

**Recursion** is a function calling itself to solve a smaller subproblem.  
**Memoization** caches results of expensive function calls to avoid redundant computation.

In [ ]:
import functools
import sys

# Naive recursion — exponential without memoization
def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

# With lru_cache — O(n) with automatic memoization
@functools.lru_cache(maxsize=None)
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(50))          # instant
print(fib.cache_info()) # CacheInfo(hits=..., misses=..., ...)

# @cache (Python 3.9+) — simpler alias for lru_cache(maxsize=None)
@functools.cache
def factorial(n):
    return 1 if n == 0 else n * factorial(n - 1)

print(factorial(20))

### Exercises — Section 4

1. Write a decorator `@validate_positive` that raises `ValueError` if any numeric argument to the decorated function is negative.
2. Write a generator `fibonacci()` that yields the Fibonacci sequence indefinitely.
3. Write `pipeline(*funcs)` that takes a series of single-argument functions and returns a function that applies them left-to-right: `pipeline(double, add_one)(5)` → `11`.

### Mini Challenge

Implement a `memoize` decorator from scratch (without `lru_cache`) that caches results in a dictionary. Handle both positional and keyword arguments correctly using a hashable cache key.

### Best Practices — Section 4

- Always use `@functools.wraps` in decorators.
- Prefer generator expressions over list comprehensions when you only iterate once.
- Use `@functools.cache` (3.9+) or `@lru_cache` for expensive recursive functions.
- Prefer `partial()` over lambda for partial application — it is more readable and introspectable.
- Keep lambdas short; if it needs more than one expression, use `def`.

### Common Mistakes — Section 4

- Forgetting `nonlocal` when mutating a closure variable — causes `UnboundLocalError`.
- Late binding in closures: `[lambda x=i: x for i in range(3)]` vs `[lambda: i for i in range(3)]` — the latter all return `2`.
- Calling `next()` on an exhausted generator raises `StopIteration`.
- Decorating without `functools.wraps` breaks `__name__`, `__doc__`, and `help()`.

### Performance Notes — Section 4

- `lru_cache` is thread-safe and implemented in C — very fast.
- Generators have negligible overhead per `yield` — prefer them for large datasets.
- Avoid deep recursion: Python's default stack depth is 1000 (`sys.getrecursionlimit()`). Prefer iterative solutions for depth > a few hundred.

### Summary — Section 4

- `*args`/`**kwargs` enable flexible, generic function signatures.
- Lambdas are concise anonymous functions — use with `sorted()`, `map()`, `filter()`.
- Higher-order functions (`map`, `filter`, `partial`) compose behavior without loops.
- Closures capture enclosing scope; factory functions create configurable functions.
- Decorators wrap functions transparently — always use `@functools.wraps`.
- Generators produce values lazily — ideal for streaming and large datasets.
- Memoization with `@cache` converts exponential recursion to linear time.

In [ ]:
# Exercise 1 — @validate_positive
# YOUR CODE HERE
# Exercise 2 — infinite fibonacci generator
def fibonacci():
    # YOUR CODE HERE
    pass
# Exercise 3 — pipeline
def pipeline(*funcs):
    # YOUR CODE HERE
    pass
# Mini Challenge — memoize from scratch
def memoize(func):
    # YOUR CODE HERE
    pass

---
# Section 5 — Object-Oriented Programming

## 5.1 Classes and Objects

### Concept

A **class** is a blueprint for objects. An **object** (instance) is a concrete realization of that blueprint.  
Python's object model: everything is an object, including classes themselves.

### Technical Deep Dive

- `__init__` is the **initializer** (not a constructor — `__new__` creates the object).
- `self` is a reference to the current instance, passed automatically.
- **Instance attributes** live on the object; **class attributes** are shared across all instances.

In [ ]:
class BankAccount:
    """A simple bank account."""
    interest_rate = 0.05   # class attribute — shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner         # instance attributes
        self.balance = balance
        self._transactions = []    # private by convention

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.balance += amount
        self._transactions.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount
        self._transactions.append(("withdraw", amount))

    def statement(self):
        return f"Account({self.owner}): ${self.balance:.2f}"


acc = BankAccount("Alice", 1000)
acc.deposit(500)
acc.withdraw(200)
print(acc.statement())
print(acc._transactions)
print(BankAccount.interest_rate)    # class attribute via class
print(acc.interest_rate)            # also accessible via instance

## 5.2 Methods: Instance, Class, Static

| Method type | Decorator | First param | Use when |
|-------------|-----------|-------------|----------|
| Instance | (none) | `self` | Needs access to instance state |
| Class | `@classmethod` | `cls` | Needs access to class, not instance (e.g. alternative constructors) |
| Static | `@staticmethod` | (none) | Utility — logically belongs to the class but needs neither |

In [ ]:
class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        if celsius < self.ABSOLUTE_ZERO_C:
            raise ValueError("Below absolute zero")
        self._celsius = celsius

    # Instance method
    def to_fahrenheit(self):
        return self._celsius * 9/5 + 32

    # Class method — alternative constructor
    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        return cls((fahrenheit - 32) * 5/9)

    @classmethod
    def from_kelvin(cls, kelvin):
        return cls(kelvin - 273.15)

    # Static method — utility, no state needed
    @staticmethod
    def is_valid_celsius(value):
        return value >= Temperature.ABSOLUTE_ZERO_C

    def __repr__(self):
        return f"Temperature({self._celsius:.2f}°C)"


t1 = Temperature(100)
t2 = Temperature.from_fahrenheit(212)
t3 = Temperature.from_kelvin(373.15)

print(t1, t1.to_fahrenheit())
print(t2)
print(t3)
print(Temperature.is_valid_celsius(-300))  # False

## 5.3 Inheritance and `super()`

### Concept

Inheritance lets a subclass **reuse and extend** a parent class.  
`super()` calls the parent's method, essential for cooperative inheritance.

In [ ]:
class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def speak(self):
        return f"{self.name} makes a sound"

    def __repr__(self):
        return f"{self.species}('{self.name}')"


class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, species="Dog")
        self.breed = breed

    def speak(self):        # override
        return f"{self.name} says Woof!"

    def fetch(self, item):
        return f"{self.name} fetches the {item}"


class GuideDog(Dog):
    def __init__(self, name, breed, owner):
        super().__init__(name, breed)
        self.owner = owner

    def speak(self):
        base = super().speak()    # call Dog.speak()
        return f"{base} (Guide dog for {self.owner})"


animals = [Animal("Cat", "Felis"), Dog("Rex", "Labrador"), GuideDog("Buddy", "Golden", "Alice")]
for a in animals:
    print(a.speak())

print(isinstance(animals[2], Dog))    # True
print(isinstance(animals[2], Animal)) # True

## 5.4 Multiple Inheritance and MRO

### Concept

Python supports **multiple inheritance**. The **Method Resolution Order (MRO)** — determined by the C3 linearization algorithm — defines which class's method is called.  
Use `ClassName.__mro__` or `ClassName.mro()` to inspect it.

In [ ]:
class Flyable:
    def move(self):
        return "flying"

class Swimmable:
    def move(self):
        return "swimming"

class Duck(Flyable, Swimmable):   # Flyable listed first
    def move(self):
        return f"{super().move()} and swimming"   # cooperative

d = Duck()
print(d.move())           # flying and swimming
print(Duck.__mro__)       # MRO chain

# Mixin pattern — the right way to use multiple inheritance
class LogMixin:
    def log(self, msg):
        print(f"[{self.__class__.__name__}] {msg}")

class JsonMixin:
    def to_json(self):
        import json
        return json.dumps(self.__dict__)

class User(LogMixin, JsonMixin):
    def __init__(self, name, email):
        self.name = name
        self.email = email

u = User("Alice", "alice@x.com")
u.log("User created")
print(u.to_json())

## 5.5 Encapsulation — Properties

### Concept

`@property` turns a method into a managed attribute — you get validation and computed values while keeping clean dot-access syntax.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius   # uses the setter

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value

    @property
    def area(self):            # computed, read-only property
        import math
        return math.pi * self._radius ** 2

    @property
    def diameter(self):
        return self._radius * 2

    @diameter.setter
    def diameter(self, value):
        self.radius = value / 2


c = Circle(5)
print(c.radius, c.area, c.diameter)
c.diameter = 20
print(c.radius)   # 10.0
try:
    c.radius = -1
except ValueError as e:
    print(e)

## 5.6 Magic (Dunder) Methods

### Concept

**Dunder methods** (`__name__`) define how objects behave with built-in operations: `len()`, `+`, `[]`, `str()`, iteration, comparison, and more.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # String representations
    def __repr__(self):            # for developers: repr(v)
        return f"Vector({self.x}, {self.y})"

    def __str__(self):             # for users: str(v) / print(v)
        return f"({self.x}, {self.y})"

    # Arithmetic operators
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):     # Vector * scalar
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):    # scalar * Vector
        return self.__mul__(scalar)

    # Comparison
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __abs__(self):             # abs(v) -> magnitude
        return (self.x**2 + self.y**2) ** 0.5

    def __len__(self):             # len(v)
        return 2

    def __iter__(self):            # for val in v
        yield self.x
        yield self.y

    def __getitem__(self, idx):    # v[0], v[1]
        return (self.x, self.y)[idx]


v1, v2 = Vector(1, 2), Vector(3, 4)
print(v1 + v2)         # (4, 6)
print(v1 * 3)          # (3, 6)
print(3 * v2)          # (9, 12)
print(abs(v2))         # 5.0
print(list(v1))        # [1, 2]
print(v1[0], v1[1])    # 1 2
print(v1 == Vector(1, 2))  # True

## 5.7 Abstract Classes

### Concept

Abstract classes define an **interface contract** — subclasses must implement all `@abstractmethod` methods. You cannot instantiate an abstract class directly.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        ...

    @abstractmethod
    def perimeter(self) -> float:
        ...

    def describe(self):    # concrete method — shared by all
        return f"{self.__class__.__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)


class Circle(Shape):
    import math as _math
    def __init__(self, r):
        self.r = r

    def area(self):
        import math
        return math.pi * self.r ** 2

    def perimeter(self):
        import math
        return 2 * math.pi * self.r


shapes = [Rectangle(4, 5), Circle(3)]
for s in shapes:
    print(s.describe())

try:
    Shape()   # cannot instantiate abstract class
except TypeError as e:
    print(e)

## 5.8 Dataclasses

### Concept

`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` (and optionally `__lt__`, `__hash__`, frozen immutability) from field annotations — eliminating boilerplate.

In [ ]:
from dataclasses import dataclass, field, asdict, astuple
from typing import List

@dataclass
class Point:
    x: float
    y: float

    def distance_to(self, other: "Point") -> float:
        return ((self.x - other.x)**2 + (self.y - other.y)**2) ** 0.5


@dataclass(order=True, frozen=True)   # frozen = immutable, hashable
class Version:
    major: int
    minor: int
    patch: int = 0

    def __str__(self):
        return f"{self.major}.{self.minor}.{self.patch}"


@dataclass
class Team:
    name: str
    members: List[str] = field(default_factory=list)   # mutable default!


p1, p2 = Point(0, 0), Point(3, 4)
print(p1, p2)
print(p1.distance_to(p2))    # 5.0

v1, v2 = Version(1, 2, 3), Version(1, 3, 0)
print(v1 < v2)               # True
print(sorted([v2, v1]))

team = Team("Backend")
team.members.append("Alice")
print(team)
print(asdict(p1))

### Exercises — Section 5

1. Create a `Stack` class with `push(item)`, `pop()`, `peek()`, `is_empty()`, and `__len__`. Use dunder methods so `len(stack)` and `str(stack)` work correctly.
2. Create a `Matrix` dataclass with `rows` and `cols`, supporting `+` and `*` (scalar) operators.
3. Create an abstract class `Serializable` with abstract methods `to_dict()` and `from_dict(cls, data)`. Implement it in a `Product` class.

### Mini Challenge

Implement a `LinkedList` class with `append(value)`, `__iter__`, `__len__`, and `__repr__`. The list should be iterable so `list(ll)` works, and `for item in ll` works.

### Best Practices — Section 5

- Use `@dataclass` for data-holding classes — eliminates boilerplate.
- Use `frozen=True` for immutable value objects (can be used as dict keys).
- Always implement `__repr__` — it makes debugging much easier.
- Use mixins for reusable behavior (`LogMixin`, `JsonMixin`) — keep them small and focused.
- Prefer composition over deep inheritance hierarchies.

### Common Mistakes — Section 5

- Using `field(default_factory=list)` is required for mutable defaults in `@dataclass` — never `members: list = []`.
- Forgetting `super().__init__()` in multi-inheritance breaks cooperative MRO.
- `__eq__` without `__hash__` makes objects unhashable — Python sets `__hash__ = None` automatically when you define `__eq__`.

### Summary — Section 5

- Classes bundle data (attributes) and behavior (methods) into reusable blueprints.
- `@classmethod` for alternative constructors; `@staticmethod` for utilities.
- Inheritance + `super()` enable code reuse across a hierarchy.
- `@property` provides validated, computed attributes with clean syntax.
- Dunder methods integrate custom objects with Python's built-in operations.
- Abstract classes enforce contracts on subclasses.
- `@dataclass` eliminates boilerplate for plain data classes.

In [ ]:
# Exercise 1 — Stack
class Stack:
    # YOUR CODE HERE
    pass
# Mini Challenge — LinkedList
class Node:
    # YOUR CODE HERE
    pass

---
# Section 6 — Intermediate Concepts

## 6.1 Exception Handling

### Concept

Exceptions are objects that signal errors. Python uses `try/except/else/finally` to handle them gracefully.  
Custom exceptions enable expressive error hierarchies specific to your domain.

### Technical Deep Dive

- `else` runs only if **no exception** was raised in `try`.
- `finally` always runs — use for cleanup (closing resources, releasing locks).
- Catch specific exceptions, never bare `except:` — it silences even `KeyboardInterrupt`.
- `raise ... from ...` chains exceptions to preserve context.

In [ ]:
# try / except / else / finally
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    except TypeError as e:
        print(f"Type error: {e}")
        return None
    else:
        print(f"Success: {a} / {b} = {result}")
        return result
    finally:
        print("--- always runs ---")

safe_divide(10, 2)
safe_divide(10, 0)
safe_divide(10, "x")

In [ ]:
# Custom exceptions
class AppError(Exception):
    """Base for all application errors."""

class ValidationError(AppError):
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"[{field}] {message}")

class NotFoundError(AppError):
    def __init__(self, resource, id):
        super().__init__(f"{resource} with id={id} not found")
        self.resource = resource
        self.id = id


def get_user(user_id):
    users = {1: "Alice", 2: "Bob"}
    if user_id not in users:
        raise NotFoundError("User", user_id)
    return users[user_id]

try:
    print(get_user(99))
except NotFoundError as e:
    print(f"Caught: {e} | resource={e.resource}")
except AppError as e:
    print(f"App error: {e}")

# Exception chaining
try:
    try:
        int("abc")
    except ValueError as e:
        raise ValidationError("age", "must be a number") from e
except ValidationError as e:
    print(e)
    print(f"Caused by: {e.__cause__}")

## 6.2 Context Managers

### Concept

A **context manager** wraps setup and teardown around a block of code using `with`.  
Guarantees cleanup even when exceptions occur — the `__exit__` method is always called.

In [ ]:
# Class-based context manager
class Timer:
    import time as _time

    def __enter__(self):
        import time
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        self.elapsed = time.perf_counter() - self._start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False   # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))
print(f"Result: {total}")

In [ ]:
from contextlib import contextmanager, suppress

# Generator-based context manager — much simpler
@contextmanager
def temp_file(path):
    import os
    f = open(path, "w")
    try:
        yield f
    finally:
        f.close()
        if os.path.exists(path):
            os.remove(path)
        print("Cleaned up")

with temp_file("/tmp/test_ctx.txt") as f:
    f.write("hello")

# suppress — silently ignore specific exceptions
with suppress(FileNotFoundError):
    import os
    os.remove("/tmp/does_not_exist.txt")
print("No crash even though file was missing")

## 6.3 File I/O — Text, Binary, CSV, JSON

### Concept

Python has built-in support for file I/O. Always use `with` for file operations — it guarantees the file handle is closed.

In [ ]:
import csv
import json
from pathlib import Path

# Text files
path = Path("/tmp/demo_io.txt")
path.write_text("line 1\nline 2\nline 3")
print(path.read_text())

# Append mode
with path.open("a") as f:
    f.write("\nline 4")

# CSV write / read
csv_path = Path("/tmp/demo.csv")
rows = [{"name": "Alice", "score": 95}, {"name": "Bob", "score": 87}]

with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerows(rows)

with csv_path.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(dict(row))

# JSON write / read
data = {"users": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]}
json_path = Path("/tmp/demo.json")
json_path.write_text(json.dumps(data, indent=2))

loaded = json.loads(json_path.read_text())
print(loaded["users"][0])

## 6.4 Modules and Packages

### Concept

A **module** is any `.py` file. A **package** is a directory with `__init__.py`.  
Python resolves imports via `sys.path`. Relative imports use `.` notation within a package.

In [ ]:
# Import styles
import os                          # import full module
import os.path as osp              # alias
from pathlib import Path           # import specific name
from collections import Counter, defaultdict   # multiple names

# Inspecting a module
import math
print(dir(math))                   # all names in module
print(math.__file__)               # module file location
print(math.__doc__[:80])           # first 80 chars of docstring

# __name__ == '__main__' pattern
# When a module is run directly, __name__ == '__main__'
# When imported, __name__ == the module name
# This is how scripts guard their main code:
# if __name__ == '__main__':
#     main()

## 6.5 Regular Expressions

### Concept

Regular expressions (regex) are patterns for matching, searching, and replacing text.  
Python's `re` module provides full regex support. Compile patterns for reuse with `re.compile()`.

In [ ]:
import re

text = "Contact: alice@example.com or bob.smith@company.org for support."

# Core functions
email_pattern = r"[\w.+-]+@[\w-]+\.[\w.]+"

# search() — first match
match = re.search(email_pattern, text)
if match:
    print("First:", match.group())

# findall() — all matches
emails = re.findall(email_pattern, text)
print("All:", emails)

# Groups
pattern = re.compile(r"(\d{4})-(\d{2})-(\d{2})")   # date
m = pattern.match("2024-03-15 is the date")
if m:
    print(m.group(0), m.group(1), m.group(2), m.group(3))

# Named groups
pattern = re.compile(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})")
m = pattern.search("Today: 2024-03-15")
if m:
    print(m.groupdict())

# sub() — replace
result = re.sub(r"\b(\w+)@\S+", r"[REDACTED]", text)
print(result)

## 6.6 Iterators and the Iterator Protocol

### Concept

An **iterable** is anything you can `for`-loop over. It implements `__iter__()`.  
An **iterator** additionally implements `__next__()` — it produces values one at a time.  
Calling `iter(obj)` gives you an iterator; `next(iterator)` advances it.

In [ ]:
# Custom iterator — range-like
class CountUp:
    def __init__(self, start, stop, step=1):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self        # iterator returns itself

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

for n in CountUp(0, 10, 2):
    print(n, end=" ")   # 0 2 4 6 8
print()

# Works with all iterator consumers
print(list(CountUp(1, 6)))   # [1, 2, 3, 4, 5]
print(sum(CountUp(1, 101)))  # 5050

## 6.7 Type Hints

### Concept

Type hints (PEP 484) are **optional annotations** that document expected types.  
They are not enforced at runtime but enable static analysis tools (`mypy`, `pyright`) and improve IDE support.

In [ ]:
from typing import Optional, Union, List, Dict, Tuple, Callable, TypeVar

# Basic annotations
def greet(name: str, times: int = 1) -> str:
    return (name + " ") * times

# Optional = Union[X, None]
def find_user(user_id: int) -> Optional[str]:
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

# Complex types
def process(
    data: List[Dict[str, Union[int, str]]],
    callback: Callable[[str], None],
) -> Tuple[int, List[str]]:
    results = []
    for item in data:
        name = str(item.get("name", ""))
        callback(name)
        results.append(name)
    return len(results), results

# TypeVar for generic functions (Python 3.11 TypeVarTuple available)
T = TypeVar("T")

def first(items: List[T]) -> Optional[T]:
    return items[0] if items else None

print(first([1, 2, 3]))          # 1 (T inferred as int)
print(first(["a", "b"]))          # 'a' (T inferred as str)

# Python 3.10+ union shorthand: str | None instead of Optional[str]
def modern(x: int | str | None) -> str:
    return str(x) if x is not None else "none"

print(modern(None), modern(42), modern("hi"))

## 6.8 `pathlib` and `os`

### Concept

`pathlib.Path` is the modern, object-oriented API for filesystem paths — prefer it over `os.path`.  
`os` and `os.path` are still useful for process-level operations.

In [ ]:
from pathlib import Path
import os

# Path construction — OS-agnostic
base = Path("/tmp")
subdir = base / "myproject" / "data"
file  = subdir / "results.csv"

# Path properties
p = Path("/tmp/demo.csv")
print(p.name)          # demo.csv
print(p.stem)          # demo
print(p.suffix)        # .csv
print(p.parent)        # /tmp
print(p.exists())
print(p.is_file())

# Create directories
output = Path("/tmp/course_output")
output.mkdir(parents=True, exist_ok=True)

# Glob — find files
for f in Path("/tmp").glob("*.txt"):
    print(f)

# Rename / move
src = output / "a.txt"
src.write_text("hello")
dst = output / "b.txt"
src.rename(dst)
print(dst.read_text())

# os — environment, process
print(os.environ.get("HOME", "N/A"))
print(os.getcwd())
print(os.cpu_count())

### Exercises — Section 6

1. Write a context manager `@contextmanager` called `working_directory(path)` that temporarily changes the current working directory and restores it on exit.
2. Write a function `parse_log_line(line: str) -> dict | None` using regex that parses lines like `"2024-03-15 ERROR [auth] Invalid token"` into `{date, level, module, message}`.
3. Write a typed function `group_by(items: list[T], key: Callable[[T], K]) -> dict[K, list[T]]` that groups items by a key function.

### Mini Challenge

Write a `retry` context manager using `@contextmanager` that retries the block up to `n` times on a given exception type:

```python
with retry(times=3, on=ValueError):
    risky_operation()
```

### Best Practices — Section 6

- Always `raise` specific exceptions — never catch bare `Exception` unless at the top level.
- Use `raise X from Y` to preserve exception chains.
- Use `pathlib.Path` for all file paths — it is cross-platform and object-oriented.
- Compile regex patterns with `re.compile()` when reusing them in loops.
- Annotate all public function signatures — use `mypy --strict` in CI.

### Common Mistakes — Section 6

- `except Exception` silences bugs; catch the most specific exception.
- `open()` without `with` can leak file handles on exceptions.
- `os.path.join` instead of `pathlib` — still works but less readable.
- Mutable default in type hints: `def f(items: list = [])` — the annotation doesn't prevent the mutation bug.

### Summary — Section 6

- `try/except/else/finally` enables fine-grained error handling; use custom exceptions for domain errors.
- Context managers (`with`) guarantee cleanup code runs regardless of exceptions.
- `pathlib.Path` is the modern, readable API for all filesystem operations.
- `re` provides full regex support; compile patterns for performance.
- Iterators and the iterator protocol power all Python iteration — `for`, `list()`, `sum()`, etc.
- Type hints are documentation + tooling — use them on all public APIs.

In [ ]:
import os
import re
from contextlib import contextmanager
from typing import Callable, TypeVar

# Exercise 1 — working_directory context manager
@contextmanager
def working_directory(path):
    original = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(original)

print("Before:", os.getcwd())
with working_directory("/tmp"):
    print("Inside:", os.getcwd())
print("After:", os.getcwd())

# Exercise 2 — parse_log_line
LOG_PATTERN = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2})\s+"
    r"(?P<level>\w+)\s+"
    r"\[(?P<module>\w+)\]\s+"
    r"(?P<message>.+)"
)

def parse_log_line(line: str) -> dict | None:
    m = LOG_PATTERN.match(line.strip())
    return m.groupdict() if m else None

print(parse_log_line("2024-03-15 ERROR [auth] Invalid token"))

# Exercise 3 — group_by
T = TypeVar("T")
K = TypeVar("K")

def group_by(items: list, key: Callable) -> dict:
    from collections import defaultdict
    result = defaultdict(list)
    for item in items:
        result[key(item)].append(item)
    return dict(result)

words = ["apple", "ant", "banana", "avocado", "blueberry"]
print(group_by(words, lambda w: w[0]))

# Mini Challenge — retry context manager
@contextmanager
def retry(times=3, on=Exception):
    for attempt in range(1, times + 1):
        try:
            yield
            break   # success — exit loop
        except on as e:
            print(f"Attempt {attempt}/{times} failed: {e}")
            if attempt == times:
                raise

import random
random.seed(42)
try:
    with retry(times=4, on=ValueError):
        if random.random() < 0.8:
            raise ValueError("bad luck")
        print("Success!")
except ValueError:
    print("All retries exhausted")

---

## End of Part 2

**Next: Part 3** will cover:
- Section 7 — Advanced Python (metaclasses, descriptors, `asyncio`, GIL, C extensions)
- Section 8 — Standard Library Deep Dive (`logging`, `unittest`, `argparse`, `pydantic`)
- Section 9 — Expert-Level Insights (design patterns, profiling, packaging)
- Section 10 — Real-World Case Studies

---
*Python Course — Part 2 | Sections 4-6*